# DocBenchTR run 3: big-model trees, fair reranking, three repeats

**Budget estimate: 4-8 A100 hours; approximately 60-120 compute units if your assigned rate is 15 CU/hour.** This is a planning range, not a measured run-3 runtime or a fixed Colab price. Previous qwen3.8:27b/64k logs average 27.3 s for Tree Search and 47.2 s for the default agent: repeating those alone takes 2.24 hours. Allow additional time for building 500 pages of trees, 108 top-20 reranks, downloads and long reasoning outputs. Actual cost = connected hours * the rate in Colab **View resources**; [Colab availability/limits vary](https://research.google.com/colaboratory/faq.html). Disconnect when finished.

Runtime > Change runtime type > **A100 GPU**, then Runtime > **Run all**. Approve Drive access. In the source cell, upload **docbenchtr-run3.bundle** when prompted (the commit is local and need not be pushed). The source bundle is saved to Drive for resume. The notebook checks out the full immutable SHA below, never `main`.

Protocol: `qwen3.8:27b` for **index + search**, `num_ctx=65536`, `num_predict=32768`, temperature 0. Methods: BM25, Turkish-stemmed BM25, Dense, Hybrid, PageIndex Tree Search, PageIndex agentic **default instructions**, Hybrid + LLM rerank. Each of the 36 questions runs with all seven methods on each of **three** repeats. Trees are built once and reused across repeats; this measures search variability conditional on those trees, not indexing variability.

Native Ollama calls avoid the recorded LiteLLM tool-argument streaming failures. The agent keeps PageIndex's default instructions/tools. A conservative context guard preserves the question and records any tool-result shortening. Provider errors remain visible and failed attempts score zero. Reranking uses full Hybrid page texts and an exact top-20 permutation; failures never silently fall back to Hybrid.

Results and resumable per-question checkpoints go directly to Drive by **index model / search model / context / repeat**. `aggregate.json` gives repeat means, sample SD, per-question majority, and paired question-level statistics vs Hybrid (original + expanded gold). Three repeats do not turn 36 questions into 108 independent observations. Larger human-reviewed questions and unified PDF extraction remain future work.

To resume after a disconnect, reconnect to an A100 and Run all with the same parameters. Source, trees and completed query rows are reused from Drive. Stopping the runner cell terminates its PageIndex worker too; the unfinished query runs again on resume.

In [ ]:
# 0. Parameters. Immutable code revision; no branch checkout or git pull.
REPO_COMMIT = '77c6dbb6bf8cf60890be54a057491de4f6231755'  # tested implementation commit
SOURCE_MODE = 'bundle'  # 'bundle' for local, unpushed work; 'github' after that SHA is published
INDEX_MODEL = 'qwen3.8:27b'
SEARCH_MODEL = 'qwen3.8:27b'
NUM_CTX = 65536
NUM_PREDICT = 32768
REPEATS = 3
OLLAMA_VERSION = '0.40.5'
CU_PER_HOUR_ESTIMATE = 15.0  # replace with the assigned rate shown in View resources
EXPERIMENT = 'run3'  # change this to start a distinct experiment; same value resumes checkpoints
assert len(REPO_COMMIT) == 40 and all(c in '0123456789abcdef' for c in REPO_COMMIT)
assert REPO_COMMIT != '0' * 40, 'Set the tested immutable commit before running.'
print(f'Planning range: 4-8 hours, {4*CU_PER_HOUR_ESTIMATE:.0f}-{8*CU_PER_HOUR_ESTIMATE:.0f} CU')

In [ ]:
# 1. Require A100 and mount Drive before any expensive setup.
import os, sys, json, time, subprocess, shutil, urllib.request, signal
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
GPU_INFO = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip()
assert 'A100' in GPU_INFO, f'Select Runtime > Change runtime type > A100 GPU; got {GPU_INFO}'
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'Drive authorization is required; rerun this cell.'
DRIVE_ROOT = Path('/content/drive/MyDrive/docbenchtr_colab')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
probe = DRIVE_ROOT / '.run3_write_probe'
probe.write_text('ok', encoding='utf-8')
assert probe.read_text() == 'ok'
probe.unlink()
print(GPU_INFO, '\nDrive:', DRIVE_ROOT)

In [ ]:
# 2. Fetch the pinned source, including local unpushed commits via a Git bundle.
REPO_URL = 'https://github.com/umitaltinozzz/docbenchtr.git'
REPO = Path('/content') / f'docbenchtr_run3_{REPO_COMMIT[:12]}'
if SOURCE_MODE == 'bundle':
    source_dir = DRIVE_ROOT / 'source_bundles'
    source_dir.mkdir(exist_ok=True)
    bundle = source_dir / f'{REPO_COMMIT}.bundle'
    if not bundle.exists():
        from google.colab import files
        print('Choose the local .run3/docbenchtr-run3.bundle file.')
        uploaded = files.upload()
        matches = [name for name in uploaded if name.endswith('.bundle')]
        assert len(matches) == 1, 'Upload exactly one Git bundle.'
        bundle.write_bytes(uploaded[matches[0]])
        del uploaded
    if not (REPO / '.git').exists():
        subprocess.run(['git', 'clone', '--no-checkout', str(bundle), str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'bundle', 'verify', str(bundle)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'fetch', str(bundle), 'HEAD'], check=True)
else:
    assert SOURCE_MODE == 'github'
    if not (REPO / '.git').exists():
        subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REPO_COMMIT], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert COMMIT == REPO_COMMIT
subprocess.run(['git', '-C', str(REPO), 'diff', '--exit-code', 'HEAD', '--', 'docbenchtr', 'pyproject.toml', 'uv.lock'], check=True)
assert (REPO / 'docbenchtr/run_repeats.py').is_file()
print('Pinned commit:', COMMIT)

In [ ]:
# 3. Frozen Python dependencies, Poppler, and CPU-only tests (no model calls).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'poppler-utils', 'zstd'], check=True)
subprocess.run(['uv', 'sync', '--frozen', '--python', '3.12', '--group', 'dev'], cwd=REPO, check=True)
PY = str(REPO / '.venv/bin/python')
subprocess.run([PY, '-m', 'pytest', '-q'], cwd=REPO, env=dict(os.environ, MPLBACKEND='Agg', LITELLM_LOCAL_MODEL_COST_MAP='True'), check=True)
subprocess.run([PY, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__)'], cwd=REPO, check=True)
EXTRACTOR_VERSION = subprocess.run(['pdftotext', '-v'], capture_output=True, text=True)
EXTRACTOR_VERSION = EXTRACTOR_VERSION.stdout + EXTRACTOR_VERSION.stderr

In [ ]:
# 4. Pinned Ollama on a dedicated port; never attach to an unknown existing server.
HOST = '127.0.0.1:11435'
BASE_URL = 'http://' + HOST
server_env = dict(os.environ, OLLAMA_HOST=HOST, OLLAMA_CONTEXT_LENGTH=str(NUM_CTX),
                  OLLAMA_MODELS=str(DRIVE_ROOT / 'ollama_models'), OLLAMA_MAX_LOADED_MODELS='1',
                  OLLAMA_NUM_PARALLEL='1', OLLAMA_FLASH_ATTENTION='1')
Path(server_env['OLLAMA_MODELS']).mkdir(exist_ok=True)
SERVER_LOG = DRIVE_ROOT / f'ollama_{EXPERIMENT}_{COMMIT[:12]}.log'
try:
    with urllib.request.urlopen(BASE_URL + '/api/version', timeout=2) as response:
        running_version = json.load(response)['version']
except Exception:
    running_version = None
if running_version is None:
    installer = Path('/tmp/docbenchtr-ollama-install.sh')
    urllib.request.urlretrieve('https://ollama.com/install.sh', installer)
    with open('/tmp/docbenchtr-ollama-install.log', 'w') as install_log:
        subprocess.run(['sh', str(installer)], env=dict(os.environ, OLLAMA_VERSION=OLLAMA_VERSION),
                       stdout=install_log, stderr=subprocess.STDOUT, check=True)
    with SERVER_LOG.open('a') as server_log:
        OLLAMA_PROCESS = subprocess.Popen(['ollama', 'serve'], env=server_env,
                                         stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(90):
        if OLLAMA_PROCESS.poll() is not None:
            raise RuntimeError(f'Ollama exited; read {SERVER_LOG}')
        try:
            with urllib.request.urlopen(BASE_URL + '/api/version', timeout=2) as response:
                running_version = json.load(response)['version']
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError(f'Ollama did not become ready; read {SERVER_LOG}')
    OWN_SERVER_SETTINGS = server_env.copy()
else:
    assert globals().get('OWN_SERVER_SETTINGS') == server_env, 'Port 11435 belongs to a previous/unknown session. Restart this runtime and Run all.'
assert running_version == OLLAMA_VERSION, (running_version, OLLAMA_VERSION)
print('Ollama', running_version, 'ready; log:', SERVER_LOG)

In [ ]:
# 5. Pull models, record weights and environment. All generation settings are per call.
# No Modelfile rewrite of model tags; index and search use the requested original models.
MODEL_RECORDS = {}
for model in dict.fromkeys([INDEX_MODEL, SEARCH_MODEL]):
    subprocess.run(['ollama', 'pull', model], env=server_env, check=True)
    request = urllib.request.Request(BASE_URL + '/api/show', data=json.dumps({'model': model}).encode(),
                                     headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(request, timeout=60) as response:
        MODEL_RECORDS[model] = json.load(response)
with urllib.request.urlopen(BASE_URL + '/api/tags', timeout=30) as response:
    MODEL_TAGS = json.load(response)
# Include code SHA and experiment label so a different revision cannot contaminate a resume.
OUTPUT_ROOT = DRIVE_ROOT / EXPERIMENT / COMMIT / 'results'
CACHE_ROOT = DRIVE_ROOT / EXPERIMENT / COMMIT / 'pageindex_models'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
run_env = dict(os.environ, DOCBENCHTR_INDEX_MODEL=INDEX_MODEL, DOCBENCHTR_OLLAMA_MODEL=SEARCH_MODEL,
               DOCBENCHTR_NUM_CTX=str(NUM_CTX), DOCBENCHTR_NUM_PREDICT=str(NUM_PREDICT),
               DOCBENCHTR_OLLAMA_BASE_URL=BASE_URL, DOCBENCHTR_CACHE_ROOT=str(CACHE_ROOT),
               MPLBACKEND='Agg', PYTHONUNBUFFERED='1', LITELLM_LOCAL_MODEL_COST_MAP='True')
DIGESTS = {m['name']: m['digest'] for m in MODEL_TAGS['models'] if m['name'] in MODEL_RECORDS}
assert len(DIGESTS) == len(MODEL_RECORDS)
weights_path = OUTPUT_ROOT / 'model_digests.json'
if weights_path.exists():
    assert json.loads(weights_path.read_text()) == DIGESTS, 'Model weights changed; use a new EXPERIMENT label.'
else:
    weights_path.write_text(json.dumps(DIGESTS, indent=2))
run_env['DOCBENCHTR_MODEL_DIGESTS'] = json.dumps(DIGESTS, sort_keys=True)
environment = {'commit': COMMIT, 'index_model': INDEX_MODEL, 'search_model': SEARCH_MODEL,
               'num_ctx': NUM_CTX, 'num_predict': NUM_PREDICT, 'temperature': 0, 'repeats': REPEATS,
               'gpu': GPU_INFO, 'ollama_version': running_version, 'models': MODEL_RECORDS,
               'model_tags': MODEL_TAGS, 'pdftotext_version': EXTRACTOR_VERSION,
               'server_settings': {k: v for k, v in server_env.items() if k.startswith('OLLAMA_')},
               'started_at': datetime.now(timezone.utc).isoformat()}
environment_path = OUTPUT_ROOT / 'run_environment.json'
if not environment_path.exists():
    environment_path.write_text(json.dumps(environment, indent=2))
else:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    (OUTPUT_ROOT / f'resume_environment_{stamp}.json').write_text(json.dumps(environment, indent=2))
print('Results:', OUTPUT_ROOT, '\nModel-specific trees:', CACHE_ROOT)

In [ ]:
# 6. One runner, all seven methods x three repeats, direct-to-Drive checkpoints.
# Rerun this cell to resume the same experiment; recorded failures are not cherry-picked.
RUN_LOG = OUTPUT_ROOT / 'run.log'
command = [PY, '-u', '-m', 'docbenchtr.run_repeats', '--repeats', str(REPEATS),
           '--output-root', str(OUTPUT_ROOT), '--resume']
started = time.monotonic()
proc = subprocess.Popen(command, cwd=REPO, env=run_env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
                        start_new_session=True)
try:
    with RUN_LOG.open('a', encoding='utf-8') as log:
        for line in proc.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
    return_code = proc.wait()
finally:
    if proc.poll() is None:
        # Stop the runner AND its PageIndex worker before resuming this cell.
        os.killpg(proc.pid, signal.SIGTERM)
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
    elapsed_hours = (time.monotonic() - started) / 3600
    for repeat_dir in OUTPUT_ROOT.glob('index_*/search_*/ctx*/repeat_*'):
        if not (repeat_dir / 'run_environment.json').exists():
            shutil.copy2(OUTPUT_ROOT / 'run_environment.json', repeat_dir / 'run_environment.json')
    print(f'Runner wall time this session: {elapsed_hours:.2f} h; estimated CU at entered rate: {elapsed_hours*CU_PER_HOUR_ESTIMATE:.1f}')
assert return_code == 0, f'Run stopped; inspect {RUN_LOG}. Fix infrastructure then rerun this cell to resume.'
aggregates = list(OUTPUT_ROOT.glob('index_*/search_*/ctx*/aggregate.json'))
assert len(aggregates) == 1
report = json.loads(aggregates[0].read_text())
for method, values in report['methods'].items():
    print(method, 'Recall@5 mean/SD:', values['metrics']['recall@5'],
          'failed attempts:', sum(r['failed_as_zero'] for r in values['per_repeat']))
    if values['paired_vs_hybrid']:
        paired = values['paired_vs_hybrid']['original_gold']['metrics']['recall@5']
        print('  vs Hybrid: mean difference', paired['mean_difference'],
              '95% CI', paired['bootstrap_95_ci'], 'sign-test p', paired['exact_two_sided_sign_test_p'])
print('Full paired statistics and majority votes:', aggregates[0])

In [ ]:
# 7. Download portable results + trees; Drive already contains every checkpoint.
# The source PDFs are pinned in the Git bundle; source metadata and labels are in each repeat.
package_root = OUTPUT_ROOT.parent
shutil.copy2(SERVER_LOG, package_root / 'ollama-server.log')
archive = shutil.make_archive('/content/docbenchtr_run3_results', 'zip', package_root)
from google.colab import files
files.download(archive)
for model in dict.fromkeys([INDEX_MODEL, SEARCH_MODEL]):
    subprocess.run(['ollama', 'stop', model], env=server_env, check=False)
print('Saved to Drive:', package_root)
print('After download: Runtime > Disconnect and delete runtime to stop compute-unit usage.')